# LeXi v2: Adaptive Technical Vocabulary Tutor

`LeXi`는 영어 기술 문서를 읽는 한국인 개발자를 위한 학습 에이전트다.
이 노트북은 `challenge/12` README에 맞춰, 설계와 구현을 함께 진행하는 LeXi v2의 작업 공간이다.

이번 단계에서는 우선 LeXi v2의 상태 모델과 structured output 스키마를 정리한다.
            


In [ ]:
from __future__ import annotations

import os
from datetime import datetime, timezone
from pathlib import Path
from pprint import pprint
from typing import Literal

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from pydantic import BaseModel, Field
from typing_extensions import TypedDict

load_dotenv(Path.cwd() / ".env")

if not os.getenv("GOOGLE_API_KEY"):
    raise RuntimeError("GOOGLE_API_KEY must be set in the project .env file.")

llm = init_chat_model(
    model=os.getenv("GOOGLE_GENAI_MODEL", "gemini-3-flash-preview"),
    model_provider="google_genai",
    temperature=0,
)

UTC = timezone.utc


## Foundation Building

LeXi v2는 `MessagesState` 대신 `custom state`를 사용한다.
핵심은 대화 전체를 누적하는 것이 아니라, 입력을 분류하고 학습 카드와 복습 기록을 안정적으로 관리하는 것이다.
            


In [ ]:
RouteName = Literal["paragraph", "single_term", "review", "reentry", "invalid"]
ReviewResult = Literal["correct", "wrong", "unknown"]
StudyPriority = Literal["high", "medium", "low"]


class VocabularyEntry(TypedDict):
    word: str
    lemma: str
    meaning_in_context: str
    source_sentence: str
    context_note: str
    why_it_matters: str
    study_priority: StudyPriority


class MemoryRecord(TypedDict):
    word: str
    lemma: str
    meaning_in_context: str
    source_sentence: str
    context_note: str
    created_at: str
    review_count: int
    last_reviewed_at: str | None
    last_review_result: ReviewResult | None


class TermEvidence(TypedDict):
    term: str
    source_sentences: list[str]


class ReviewState(TypedDict):
    current_word: str
    current_source_sentence: str
    expected_meaning: str
    user_answer: str
    judgment: ReviewResult
    explanation: str


class LearningState(TypedDict):
    input_text: str
    route: RouteName | None
    candidate_words: list[str]
    terms_to_study: list[str]
    term_evidence: dict[str, TermEvidence]
    vocabulary_entries: list[VocabularyEntry]
    memory_records: list[MemoryRecord]
    review_queue: list[str]
    review_state: ReviewState | None
    review_history: list[dict[str, str]]
    continue_review: bool | None
    error_message: str | None
    session_review_limit: int


class RouteDecision(BaseModel):
    route: RouteName = Field(description="Detected learning route for the current input")
    reason: str = Field(description="Short explanation for why the route was chosen")


class CandidateWords(BaseModel):
    candidate_words: list[str] = Field(
        description="Important English technical words or short terms worth studying. Maximum 5 items."
    )


class NormalizedTerms(BaseModel):
    terms_to_study: list[str] = Field(
        description="Normalized English terms to study. Maximum 5 items."
    )


class VocabularyEntryModel(BaseModel):
    word: str = Field(description="Word or phrase as it appears in the text")
    lemma: str = Field(description="Normalized lemma")
    meaning_in_context: str = Field(description="Korean meaning in the current technical context")
    source_sentence: str = Field(description="Source sentence from the original English text")
    context_note: str = Field(description="Short Korean explanation of why this meaning fits the context")
    why_it_matters: str = Field(description="Short Korean explanation of why this term matters for technical reading")
    study_priority: StudyPriority = Field(description="Study priority level for the learner")


class VocabularyEntries(BaseModel):
    vocabulary_entries: list[VocabularyEntryModel]


class ReviewJudgment(BaseModel):
    judgment: ReviewResult = Field(description="Whether the user's Korean answer is correct, wrong, or unknown")
    explanation: str = Field(description="Short Korean explanation used when the user needs feedback")


route_llm = llm.with_structured_output(RouteDecision)
candidate_llm = llm.with_structured_output(CandidateWords)
term_normalizer_llm = llm.with_structured_output(NormalizedTerms)
entry_llm = llm.with_structured_output(VocabularyEntries)
review_judge_llm = llm.with_structured_output(ReviewJudgment)


def make_initial_state(input_text: str = "") -> LearningState:
    return {
        "input_text": input_text,
        "route": None,
        "candidate_words": [],
        "terms_to_study": [],
        "term_evidence": {},
        "vocabulary_entries": [],
        "memory_records": [],
        "review_queue": [],
        "review_state": None,
        "review_history": [],
        "continue_review": None,
        "error_message": None,
        "session_review_limit": 3,
    }
            


In [ ]:
sample_state = make_initial_state("Caching reduces latency, but stale data can break correctness.")
pprint(sample_state)
print("schema_ready_at", datetime.now(UTC).isoformat())
            
